# Conformal Prediction for Implied Volatility Surfaces

Implementation of rolling window conformal prediction with Vega-weighted nonconformity scores.

**Dataset:** SPX options, 22 surfaces (Dec 2025 - Apr 2026)  
**Train:** 15 surfaces | **Test:** 7 surfaces | **Window:** K=5

## Setup

In [ ]:
import sys
import os
from pathlib import Path

sys.path.insert(0, os.path.abspath('../../..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch.utils.data import DataLoader

from volatility_smoothing.utils.options_data import WRDSOptionsDataset
from volatility_smoothing.utils.train.dataset import GNOOptionsDataset
from volatility_smoothing.utils.train.loss import Loss
from volatility_smoothing.utils.train import misc
from volatility_smoothing.utils.black_scholes import vega
from volatility_smoothing.conformal.nonconformity_scores import compute_scores_from_data
from volatility_smoothing.conformal.rolling_window import RollingWindowCalibration
from volatility_smoothing.conformal.mondrian import MondrianConformal

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (14, 8)

print("Imports complete")

## Configuration

In [ ]:
CHECKPOINT_PATH = "../../train/store/9448705/checkpoints/checkpoint_final.pt"
DATA_DIR = "../../data/openbb/spx"
CACHE_DIR = os.path.expanduser('~/.cache/opds_rolling_window')

device = torch.device("cpu")

ALPHA = 0.1
WINDOW_SIZE = 5
TRAIN_SIZE = 15
SCORE_TYPE = 'vega'

# Mondrian binning configuration
N_BINS_RHO = 4
N_BINS_Z = 4
RHO_RANGE = (0.01, 1.0)
Z_RANGE = (-1.5, 0.5)
SHRINKAGE = 0.3  # 30% regularization toward global quantile

print(f"Coverage target: {1-ALPHA:.0%}")
print(f"Window size K: {WINDOW_SIZE}")
print(f"Score type: {SCORE_TYPE}")
print(f"Mondrian bins: {N_BINS_RHO}x{N_BINS_Z}")
print(f"Regularization shrinkage: {SHRINKAGE:.1%}")

os.environ['OPDS_CACHE_DIR'] = CACHE_DIR
os.environ['OPDS_WRDS_DATA_DIR'] = os.path.abspath(DATA_DIR)
os.environ['OPDS_FILE_MODE'] = 'combined'

## Load Data

In [ ]:
dataset = WRDSOptionsDataset()
n_surfaces = len(dataset)
dates = dataset.quote_datetimes

print(f"Total surfaces: {n_surfaces}")
print(f"Date range: {dates[0]} to {dates[-1]}\n")
for i, date in enumerate(dates):
    print(f"  [{i:2d}] {date}")

In [ ]:
model, _ = misc.load_checkpoint(CHECKPOINT_PATH, device=device)
model.to(device)
model.eval()

gno_dataset = GNOOptionsDataset(dataset)
dev_loss = Loss(step_r=0.01, step_z=0.01)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")

## Train/Test Split

In [ ]:
cal_idx = list(range(TRAIN_SIZE))
test_idx = list(range(TRAIN_SIZE, n_surfaces))

print(f"Calibration ({len(cal_idx)} surfaces):")
for i in cal_idx:
    print(f"  [{i:2d}] {dates[i]}")

print(f"\nTest ({len(test_idx)} surfaces):")
for i in test_idx:
    print(f"  [{i:2d}] {dates[i]}")

## Rolling Window Calibration

In [ ]:
rolling_window = RollingWindowCalibration(window_size=WINDOW_SIZE)
quantile_history = []

print(f"Calibrating on {len(cal_idx)} surfaces (K={WINDOW_SIZE})...\n")

for idx in cal_idx:
    dataloader = DataLoader([gno_dataset[idx]], batch_size=1, collate_fn=dev_loss.collate_fn)
    
    for data, input_dict, aux in dataloader:
        input_dict = {k: v.to(device) if torch.is_tensor(v) else v for k, v in input_dict.items()}
        
        with torch.no_grad():
            output = model(**input_dict)
        
        iv_predict, iv_surface, *_ = dev_loss.read_output(output, aux)
        
        # Compute Vega-weighted scores (Score B)
        scores_b = compute_scores_from_data(
            score_type='vega',
            data=data,
            iv_predict=iv_predict,
            model=model,
            device=device,
            min_weight=1.0
        )
        
        # Flatten arrays for storage
        rho_flat = data['r'].cpu().numpy().flatten()
        z_flat = data['z'].cpu().numpy().flatten()
        
        rolling_window.add_surface(
            quote_datetime=data['quote_datetime'],
            rho=rho_flat,
            z=z_flat,
            scores=scores_b
        )
        
        # Get calibration data
        rho_cal, z_cal, scores_cal = rolling_window.get_calibration_data()
        
        # Initialize Mondrian with current window data and regularization
        mondrian = MondrianConformal(
            n_bins_rho=N_BINS_RHO,
            n_bins_z=N_BINS_Z,
            rho_range=RHO_RANGE,
            z_range=Z_RANGE,
            alpha=ALPHA,
            min_samples_per_bin=10,
            shrinkage=SHRINKAGE
        )
        
        # Calibrate Mondrian on window scores
        mondrian.calibrate(rho=rho_cal, z=z_cal, scores=scores_cal)
        
        # Get global quantile for comparison
        n_cal = len(scores_cal)
        q_level = np.ceil((n_cal + 1) * (1 - ALPHA)) / n_cal
        q_global = np.quantile(scores_cal, q_level)
        
        # Get Mondrian summary
        summary = mondrian.summary()
        
        quantile_history.append({
            'surface_idx': idx,
            'quote_datetime': data['quote_datetime'],
            'n_window': len(rolling_window),
            'n_points': len(scores_b),
            'n_cal_total': n_cal,
            'quantile_global': q_global,
            'quantile_mondrian_mean': summary['mean_bin_quantile'],
            'quantile_mondrian_min': summary['min_bin_quantile'],
            'quantile_mondrian_max': summary['max_bin_quantile'],
            'score_mean': scores_b.mean(),
            'score_std': scores_b.std(),
            'empty_bins': summary['empty_bins'],
            'sparse_bins': summary['sparse_bins']
        })
        
        window_status = "full" if rolling_window.is_full() else f"{len(rolling_window)}/{WINDOW_SIZE}"
        print(f"[{idx:2d}] {data['quote_datetime']}: {len(scores_b):>5} points, "
              f"q_global={q_global:.6f}, q_mondrian=[{summary['min_bin_quantile']:.6f}, {summary['max_bin_quantile']:.6f}], "
              f"window={window_status}")
        break


### Calibration Analysis

In [ ]:
cal_df = pd.DataFrame(quantile_history)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Quantile evolution: Global vs Mondrian
axes[0].plot(cal_df['surface_idx'], cal_df['quantile_global'], 'o-', 
                linewidth=2, markersize=6, label='Global', color='blue')
axes[0].plot(cal_df['surface_idx'], cal_df['quantile_mondrian_mean'], 's-', 
                linewidth=2, markersize=6, label='Mondrian (mean)', color='green')
axes[0].fill_between(cal_df['surface_idx'], 
                         cal_df['quantile_mondrian_min'], 
                         cal_df['quantile_mondrian_max'],
                         alpha=0.2, color='green', label='Mondrian (range)')
axes[0].axvline(WINDOW_SIZE-1, color='red', linestyle='--', alpha=0.5, label='Window full')
axes[0].set_xlabel('Surface Index')
axes[0].set_ylabel('Quantile q (90th percentile)')
axes[0].set_title('Quantile Evolution: Global vs Mondrian')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Score statistics
axes[1].plot(cal_df['surface_idx'], cal_df['score_mean'], 'o-', linewidth=2, markersize=6, color='orange')
axes[1].set_xlabel('Surface Index')
axes[1].set_ylabel('Mean Score B')
axes[1].set_title('Average Vega-Weighted Score per Surface')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Test Set Evaluation

In [ ]:
test_results = []

print(f"Testing on {len(test_idx)} held-out surfaces...\n")

for idx in test_idx:
    dataloader = DataLoader([gno_dataset[idx]], batch_size=1, collate_fn=dev_loss.collate_fn)
    
    for data, input_dict, aux in dataloader:
        input_dict = {k: v.to(device) if torch.is_tensor(v) else v for k, v in input_dict.items()}
        
        # Get calibration data from rolling window
        rho_cal, z_cal, scores_cal = rolling_window.get_calibration_data()
        
        # Initialize and calibrate Mondrian on current window with regularization
        mondrian = MondrianConformal(
            n_bins_rho=N_BINS_RHO,
            n_bins_z=N_BINS_Z,
            rho_range=RHO_RANGE,
            z_range=Z_RANGE,
            alpha=ALPHA,
            min_samples_per_bin=10,
            shrinkage=SHRINKAGE
        )
        mondrian.calibrate(rho=rho_cal, z=z_cal, scores=scores_cal)
        
        # Get global quantile for comparison
        n_cal = len(scores_cal)
        q_level = np.ceil((n_cal + 1) * (1 - ALPHA)) / n_cal
        q_global = np.quantile(scores_cal, q_level)
        
        # Get model predictions
        with torch.no_grad():
            output = model(**input_dict)
        
        iv_predict, _, *_ = dev_loss.read_output(output, aux)
        
        iv_pred_np = iv_predict.cpu().numpy()
        iv_market_np = data['implied_volatility'].cpu().numpy()
        
        # Compute vega weights
        vega_vals = vega(data['r'], data['z'], iv_predict).cpu().numpy()
        mean_vega = vega_vals.mean()
        weights = vega_vals / mean_vega
        weights = np.maximum(weights, 1.0)
        
        # Get per-point quantiles from Mondrian (flatten inputs)
        rho_test = data['r'].cpu().numpy().flatten()
        z_test = data['z'].cpu().numpy().flatten()
        q_mondrian = mondrian.get_quantiles(rho=rho_test, z=z_test)
        
        # Flatten predictions and market data
        iv_pred_flat = iv_pred_np.flatten()
        iv_market_flat = iv_market_np.flatten()
        weights_flat = weights.flatten()
        
        # Compute bands using Mondrian quantiles
        delta_iv_mondrian = q_mondrian / weights_flat
        iv_lower_mondrian = iv_pred_flat - delta_iv_mondrian
        iv_upper_mondrian = iv_pred_flat + delta_iv_mondrian
        
        # Compute bands using global quantile (for comparison)
        delta_iv_global = q_global / weights_flat
        iv_lower_global = iv_pred_flat - delta_iv_global
        iv_upper_global = iv_pred_flat + delta_iv_global
        
        # Coverage metrics
        covered_mondrian = (iv_market_flat >= iv_lower_mondrian) & (iv_market_flat <= iv_upper_mondrian)
        coverage_mondrian = covered_mondrian.mean()
        
        covered_global = (iv_market_flat >= iv_lower_global) & (iv_market_flat <= iv_upper_global)
        coverage_global = covered_global.mean()
        
        # Other metrics
        mae = np.abs(iv_pred_flat - iv_market_flat).mean()
        rmse = np.sqrt(((iv_pred_flat - iv_market_flat)**2).mean())
        
        result = {
            'surface_idx': idx,
            'quote_datetime': data['quote_datetime'],
            'n_options': len(iv_market_flat),
            'quantile_global': q_global,
            'quantile_mondrian_mean': q_mondrian.mean(),
            'quantile_mondrian_min': q_mondrian.min(),
            'quantile_mondrian_max': q_mondrian.max(),
            'coverage_global': coverage_global,
            'coverage_mondrian': coverage_mondrian,
            'mae': mae,
            'rmse': rmse,
            'band_width_mean_global': (2 * delta_iv_global).mean(),
            'band_width_mean_mondrian': (2 * delta_iv_mondrian).mean(),
            'band_width_median_global': np.median(2 * delta_iv_global),
            'band_width_median_mondrian': np.median(2 * delta_iv_mondrian),
            'n_cal_points': n_cal,
            'window_surfaces': list(rolling_window.get_surface_dates())
        }
        test_results.append(result)
        
        print(f"[{idx:2d}] {data['quote_datetime']}: "
              f"Coverage (Global)={coverage_global:.2%}, "
              f"Coverage (Mondrian)={coverage_mondrian:.2%}, "
              f"MAE={mae:.6f}")
        
        # Add test surface to rolling window for next iteration (flatten arrays)
        scores_test = compute_scores_from_data(
            score_type='vega',
            data=data,
            iv_predict=iv_predict,
            model=model,
            device=device,
            min_weight=1.0
        )
        
        rolling_window.add_surface(
            quote_datetime=data['quote_datetime'],
            rho=rho_test,  # Already flattened above
            z=z_test,      # Already flattened above
            scores=scores_test
        )
        
        break

print(f"\nTesting complete")

In [ ]:
test_df = pd.DataFrame(test_results)

# Coverage statistics
mean_coverage_global = test_df['coverage_global'].mean()
mean_coverage_mondrian = test_df['coverage_mondrian'].mean()
std_coverage_global = test_df['coverage_global'].std()
std_coverage_mondrian = test_df['coverage_mondrian'].std()
min_coverage_global = test_df['coverage_global'].min()
min_coverage_mondrian = test_df['coverage_mondrian'].min()
max_coverage_global = test_df['coverage_global'].max()
max_coverage_mondrian = test_df['coverage_mondrian'].max()

mean_mae = test_df['mae'].mean()
mean_band_width_global = test_df['band_width_mean_global'].mean()
mean_band_width_mondrian = test_df['band_width_mean_mondrian'].mean()

print("="*80)
print("COMPARISON: Global vs Mondrian Conformal Prediction")
print("="*80)

print("\nCoverage Statistics (Global):")
print(f"  Mean:   {mean_coverage_global:.2%} {'PASS' if mean_coverage_global >= 0.88 else 'FAIL'}")
print(f"  Std:    {std_coverage_global:.2%}")
print(f"  Range:  [{min_coverage_global:.2%}, {max_coverage_global:.2%}]")

print("\nCoverage Statistics (Mondrian):")
print(f"  Mean:   {mean_coverage_mondrian:.2%} {'PASS' if mean_coverage_mondrian >= 0.88 else 'FAIL'}")
print(f"  Std:    {std_coverage_mondrian:.2%}")
print(f"  Range:  [{min_coverage_mondrian:.2%}, {max_coverage_mondrian:.2%}]")

improvement = mean_coverage_mondrian - mean_coverage_global
print(f"\n  Improvement: {improvement:+.2%}")

print(f"\nPrediction Quality:")
print(f"  Mean MAE: {mean_mae:.6f}")
print(f"  Mean band width (Global):   {mean_band_width_global/2:.6f}")
print(f"  Mean band width (Mondrian): {mean_band_width_mondrian/2:.6f}")

print(f"\nPer-Surface Results:")
print(f"{'Idx':<4} {'Date':<20} {'Cov(G)':>8} {'Cov(M)':>8} {'Diff':>7} {'MAE':>10} {'q_range(M)':>20}")
print("-"*85)
for _, row in test_df.iterrows():
    diff = row['coverage_mondrian'] - row['coverage_global']
    q_range = f"[{row['quantile_mondrian_min']:.4f}, {row['quantile_mondrian_max']:.4f}]"
    print(f"{row['surface_idx']:<4} {str(row['quote_datetime'])[:19]:<20} "
          f"{row['coverage_global']:>7.2%} {row['coverage_mondrian']:>7.2%} {diff:>+6.2%} "
          f"{row['mae']:>10.6f} {q_range:>20}")

print("\n" + "="*80)
print(f"Surfaces below 90% coverage:")
print(f"  Global:   {(test_df['coverage_global'] < 0.90).sum()} surfaces")
print(f"  Mondrian: {(test_df['coverage_mondrian'] < 0.90).sum()} surfaces")
print("="*80)

## 3D Visualization

In [ ]:
test_surf_idx = 15

dataloader_viz = DataLoader([gno_dataset[test_surf_idx]], batch_size=1, collate_fn=dev_loss.collate_fn)

for data_viz, input_dict_viz, aux_viz in dataloader_viz:
    input_dict_viz = {k: v.to(device) if torch.is_tensor(v) else v for k, v in input_dict_viz.items()}
    
    rw = RollingWindowCalibration(window_size=WINDOW_SIZE)
    
    for idx in range(test_surf_idx - WINDOW_SIZE, test_surf_idx):
        dl = DataLoader([gno_dataset[idx]], batch_size=1, collate_fn=dev_loss.collate_fn)
        for d, inp, a in dl:
            inp = {k: v.to(device) if torch.is_tensor(v) else v for k, v in inp.items()}
            with torch.no_grad():
                out = model(**inp)
            iv_p, _, *_ = dev_loss.read_output(out, a)
            scores = compute_scores_from_data(
                score_type='vega', data=d, iv_predict=iv_p,
                model=model, device=device, min_weight=1.0
            )
            rw.add_surface(
                quote_datetime=d['quote_datetime'],
                rho=d['r'].cpu().numpy(),
                z=d['z'].cpu().numpy(),
                scores=scores
            )
            break
    
    _, _, scores_cal_viz = rw.get_calibration_data()
    n_cal_viz = len(scores_cal_viz)
    q_level_viz = np.ceil((n_cal_viz + 1) * (1 - ALPHA)) / n_cal_viz
    q_viz = np.quantile(scores_cal_viz, q_level_viz)
    
    with torch.no_grad():
        output_viz = model(**input_dict_viz)
    
    iv_predict_viz, _, *_ = dev_loss.read_output(output_viz, aux_viz)
    
    rho_np = data_viz['r'].cpu().numpy()
    z_np = data_viz['z'].cpu().numpy()
    iv_pred_np = iv_predict_viz.cpu().numpy()
    iv_market_np = data_viz['implied_volatility'].cpu().numpy()
    
    vega_vals_viz = vega(data_viz['r'], data_viz['z'], iv_predict_viz).cpu().numpy()
    mean_vega_viz = vega_vals_viz.mean()
    weights_viz = np.maximum(vega_vals_viz / mean_vega_viz, 1.0)
    delta_iv_viz = q_viz / weights_viz
    
    iv_lower_np = iv_pred_np - delta_iv_viz
    iv_upper_np = iv_pred_np + delta_iv_viz
    
    print(f"Surface {test_surf_idx}: {data_viz['quote_datetime']}")
    print(f"Options: {len(rho_np):,}")
    print(f"Quantile q: {q_viz:.6f}")
    print(f"Mean band width: {delta_iv_viz.mean():.6f}")
    
    break

In [ ]:
from mpl_toolkits.mplot3d import Axes3D
from scipy.interpolate import griddata

rho_flat = rho_np.flatten()
z_flat = z_np.flatten()
iv_pred_flat = iv_pred_np.flatten()
iv_lower_flat = iv_lower_np.flatten()
iv_upper_flat = iv_upper_np.flatten()
iv_market_flat = iv_market_np.flatten()

rho_grid = np.linspace(rho_flat.min(), rho_flat.max(), 30)
z_grid = np.linspace(z_flat.min(), z_flat.max(), 30)
rho_mesh, z_mesh = np.meshgrid(rho_grid, z_grid)

iv_pred_grid = griddata((rho_flat, z_flat), iv_pred_flat, (rho_mesh, z_mesh), method='cubic')
iv_lower_grid = griddata((rho_flat, z_flat), iv_lower_flat, (rho_mesh, z_mesh), method='cubic')
iv_upper_grid = griddata((rho_flat, z_flat), iv_upper_flat, (rho_mesh, z_mesh), method='cubic')
iv_market_grid = griddata((rho_flat, z_flat), iv_market_flat, (rho_mesh, z_mesh), method='cubic')

print(f"Grid resolution: {len(rho_grid)}x{len(z_grid)}")
print(f"ρ range: [{rho_flat.min():.3f}, {rho_flat.max():.3f}]")
print(f"z range: [{z_flat.min():.3f}, {z_flat.max():.3f}]")

## 3D Surfaces

In [ ]:
fig = plt.figure(figsize=(16, 6))

ax1 = fig.add_subplot(121, projection='3d')

surf1 = ax1.plot_surface(rho_mesh, z_mesh, iv_pred_grid, cmap='viridis', alpha=0.8, 
                         edgecolor='none', linewidth=0, antialiased=True)
ax1.scatter(rho_flat, z_flat, iv_market_flat, c='red', s=2, alpha=0.4, 
            label='Market IV', depthshade=True)

ax1.set_xlabel('√Maturity (ρ)')
ax1.set_ylabel('Moneyness (z)')
ax1.set_zlabel('Implied Volatility')
ax1.set_title('GNO Prediction (Point Estimates)')

ax1.invert_yaxis()
ax1.view_init(elev=20, azim=225)

cbar1 = fig.colorbar(surf1, ax=ax1, shrink=0.5, aspect=12, pad=0.08)
cbar1.set_label('IV')

ax2 = fig.add_subplot(122, projection='3d')

surf_lower = ax2.plot_surface(rho_mesh, z_mesh, iv_lower_grid, cmap='Blues', 
                               alpha=0.4, edgecolor='none', linewidth=0, antialiased=True)
surf_upper = ax2.plot_surface(rho_mesh, z_mesh, iv_upper_grid, cmap='Reds', 
                               alpha=0.4, edgecolor='none', linewidth=0, antialiased=True)
surf_pred = ax2.plot_surface(rho_mesh, z_mesh, iv_pred_grid, cmap='Greens', 
                              alpha=0.7, edgecolor='none', linewidth=0, antialiased=True)
ax2.scatter(rho_flat, z_flat, iv_market_flat, c='black', s=1.5, alpha=0.3, 
            label='Market IV', depthshade=True)

ax2.set_xlabel('√Maturity (ρ)')
ax2.set_ylabel('Moneyness (z)')
ax2.set_zlabel('Implied Volatility')
ax2.set_title('90% Conformal Bands (Vega-IV)')

ax2.invert_yaxis()
ax2.view_init(elev=20, azim=225)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='blue', alpha=0.4, label='Lower (v - δIV)'),
    Patch(facecolor='green', alpha=0.7, label='Prediction (v̂)'),
    Patch(facecolor='red', alpha=0.4, label='Upper (v + δIV)'),
]
ax2.legend(handles=legend_elements, loc='upper left', fontsize=9, framealpha=0.9)

fig.suptitle(f'Surface {test_surf_idx} ({data_viz["quote_datetime"]})', fontsize=14, y=0.98)
plt.tight_layout()
plt.show()

coverage_viz = ((iv_market_flat >= iv_lower_flat) & (iv_market_flat <= iv_upper_flat)).mean()
band_width_avg = (iv_upper_flat - iv_lower_flat).mean()

print(f"Coverage: {coverage_viz:.2%}")
print(f"Average band width: {band_width_avg:.6f}")
print(f"Quantile q: {q_viz:.6f}")

---

### Results

- Mean coverage: 89.96% (target: ≥90%)
- Window size: K=5 surfaces
- Test surfaces: 7 held-out surfaces
- Quantile range: [0.017314, 0.018535]

In [ ]:
# Interactive HTML Visualization
import plotly.graph_objects as go
from pathlib import Path
import os

# Use ABSOLUTE path to ensure file is saved
notebook_dir = Path(os.getcwd())
viz_folder = notebook_dir.parent.parent / 'visualizations'
viz_folder.mkdir(exist_ok=True)

print(f"Saving to: {viz_folder.absolute()}\n")

# Create figure
fig = go.Figure()

# Add layers with showlegend=True
fig.add_trace(go.Surface(
    x=rho_mesh, y=z_mesh, z=iv_lower_grid,
    colorscale='Blues', name='Lower Band', showscale=False, opacity=0.4,
    showlegend=True, hovertemplate='Lower: %{z:.4f}<extra></extra>'
))

fig.add_trace(go.Surface(
    x=rho_mesh, y=z_mesh, z=iv_upper_grid,
    colorscale='Reds', name='Upper Band', showscale=False, opacity=0.4,
    showlegend=True, hovertemplate='Upper: %{z:.4f}<extra></extra>'
))

fig.add_trace(go.Surface(
    x=rho_mesh, y=z_mesh, z=iv_pred_grid,
    colorscale='Greens', name='Prediction', showscale=True, opacity=0.85,
    showlegend=True, colorbar=dict(title='IV', len=0.6, x=1.1),
    hovertemplate='Prediction: %{z:.4f}<extra></extra>'
))

fig.add_trace(go.Scatter3d(
    x=rho_flat, y=z_flat, z=iv_market_flat,
    mode='markers', name='Market IV', 
    marker=dict(size=2, color='black', opacity=0.5),
    showlegend=True, hovertemplate='Market: %{z:.4f}<extra></extra>'
))

# Layout with visible legend
fig.update_layout(
    title=f'Surface {test_surf_idx} - Click Legend to Toggle',
    width=1400, height=900,
    showlegend=True,
    legend=dict(
        x=1.15, y=0.9, bgcolor='white', bordercolor='black', 
        borderwidth=2, font=dict(size=14)
    ),
    scene=dict(
        xaxis_title='√Maturity ',
        yaxis_title='Moneyness ',
        zaxis_title='Implied Volatility',
        camera=dict(eye=dict(x=1.6, y=-1.6, z=1.3))
    )
)

# Save HTML file
output_file = viz_folder / f'surface_{test_surf_idx}_interactive.html'
try:
    fig.write_html(str(output_file))
    print(f"SUCCESS! File saved to:")
    print(f"   {output_file}")
except Exception as e:
    print(f"Error saving: {e}")

# Show in notebook
fig.show()

In [ ]:
# Heatmap: Band Width Across Option Space
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import griddata

# Compute band width for each option
band_width = iv_upper_np - iv_lower_np  # v_hi(x) - v_lo(x)

# Create figure
fig, ax = plt.subplots(1, 1, figsize=(12, 8))

# Create regular grid for heatmap
z_grid_heat = np.linspace(z_np.min(), z_np.max(), 50)
rho_grid_heat = np.linspace(rho_np.min(), rho_np.max(), 50)
z_mesh_heat, rho_mesh_heat = np.meshgrid(z_grid_heat, rho_grid_heat)

# Interpolate band width onto grid
band_width_grid = griddata(
    (z_np.flatten(), rho_np.flatten()), 
    band_width.flatten(), 
    (z_mesh_heat, rho_mesh_heat), 
    method='cubic'
)

# Plot heatmap
heatmap = ax.contourf(
    z_mesh_heat, 
    rho_mesh_heat, 
    band_width_grid, 
    levels=20, 
    cmap='viridis', 
    alpha=0.9
)

# Add contour lines
contours = ax.contour(
    z_mesh_heat, 
    rho_mesh_heat, 
    band_width_grid, 
    levels=10, 
    colors='white', 
    linewidths=0.8, 
    alpha=0.5
)
ax.clabel(contours, inline=True, fontsize=9, fmt='%.4f')

ax.set_xlabel('Moneyness', fontsize=14)
ax.set_ylabel('√Maturity', fontsize=14)
ax.set_title(
    f'Conformal Band Width Distribution - Surface {test_surf_idx}\n' +
    f'Coverage: {coverage_viz:.2%} | Quantile q: {q_viz:.6f}',
    fontsize=15, 
    fontweight='bold',
    pad=20
)
ax.axvline(0, color='red', linestyle='--', linewidth=2, alpha=0.7, label='ATM (z=0)')
ax.legend(fontsize=12, loc='upper right')
ax.grid(True, alpha=0.3, linestyle=':', linewidth=0.5)

cbar = plt.colorbar(heatmap, ax=ax, pad=0.02)
cbar.set_label('Band Width (v_upper - v_lower)', fontsize=13)

plt.tight_layout()
plt.show()


